# 🚀 Runner — Exécution séquentielle des notebooks

Ce notebook lance tous les notebooks du dossier **dans l'ordre numérique** (01 → 02 → 03 → …)  
via `jupyter nbconvert --execute`. Les résultats sont sauvegardés **inplace** dans chaque fichier.

---

**Configuration disponible dans la cellule suivante :**
| Paramètre | Description | Défaut |
|---|---|---|
| `TIMEOUT` | Timeout par cellule (secondes) | `600` |
| `SKIP` | Liste à ignorer | `[]` |
| `STOP_ON_ERROR` | Stopper si un notebook échoue | `True` |

In [10]:
# ─── Configuration ───────────────────────────────────────────
TIMEOUT       = 600       # secondes par cellule
SKIP          = []        # ex: ["03", "04"] pour ignorer ces notebooks
STOP_ON_ERROR = True      # False = continue même en cas d'erreur
# ─────────────────────────────────────────────────────────────

In [11]:
import glob
import os
import subprocess
import sys
import time

NOTEBOOKS_DIR = os.path.dirname(os.path.abspath("__file__"))

# Récupère et trie les notebooks par préfixe numérique
all_notebooks = sorted(glob.glob(os.path.join(NOTEBOOKS_DIR, "*.ipynb")))

# Exclut ce runner lui-même et les notebooks dans SKIP
notebooks = [
    nb for nb in all_notebooks
    if os.path.basename(nb) != "run_all_notebooks.ipynb"
    and not any(os.path.basename(nb).startswith(s) for s in SKIP)
]

print("=" * 60)
print(f"  Pipeline — {len(notebooks)} notebook(s) détecté(s)")
print(f"  Timeout par cellule : {TIMEOUT}s")
if SKIP:
    print(f"  Notebooks ignorés   : {SKIP}")
print("=" * 60)
for nb in notebooks:
    print(f"    • {os.path.basename(nb)}")

  Pipeline — 8 notebook(s) détecté(s)
  Timeout par cellule : 600s
    • 01_EDA.ipynb
    • 02_Preprocessing.ipynb
    • 03_FeatureEngineering_Badgeuse.ipynb
    • 04_KMeans_Exploration.ipynb
    • 05_Regression_Preparation.ipynb
    • 06_Regression_Lineaire.ipynb
    • 07_Regression_Comparaison_Modeles.ipynb
    • 08_Conclusion.ipynb


In [12]:
results = []
total_start = time.time()

for i, nb_path in enumerate(notebooks, start=1):
    name = os.path.basename(nb_path)
    print(f"\n[{i}/{len(notebooks)}] ▶  {name}")
    print("-" * 60)

    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        f"--ExecutePreprocessor.timeout={TIMEOUT}",
        "--ExecutePreprocessor.kernel_name=python3",
        nb_path,
    ]

    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start

    success = result.returncode == 0
    status  = "✓ OK" if success else "✗ ERREUR"
    print(f"  {status}  —  durée : {elapsed:.1f}s")

    if not success:
        print("\n  --- Détail de l'erreur (dernières lignes) ---")
        for line in result.stderr.strip().splitlines()[-20:]:
            print(f"  {line}")

    results.append((name, success, elapsed))

    if not success and STOP_ON_ERROR:
        print("\n⚠  Arrêt du pipeline suite à une erreur.")
        print("   Mettez STOP_ON_ERROR = False pour continuer malgré les erreurs.")
        break

# ── Récapitulatif ──────────────────────────────────────────────
total_elapsed = time.time() - total_start
print("\n" + "=" * 60)
print("  RÉCAPITULATIF")
print("=" * 60)
for name, success, elapsed in results:
    icon = "✓" if success else "✗"
    print(f"  {icon}  {name:<45}  {elapsed:>7.1f}s")

nb_ok = sum(1 for _, s, _ in results if s)
nb_ko = len(results) - nb_ok
print("-" * 60)
print(f"  Résultat : {nb_ok} succès / {nb_ko} échec(s)  —  total : {total_elapsed:.1f}s")
print("=" * 60)


[1/8] ▶  01_EDA.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 8.1s

[2/8] ▶  02_Preprocessing.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 4.4s

[3/8] ▶  03_FeatureEngineering_Badgeuse.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 5.7s

[4/8] ▶  04_KMeans_Exploration.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 16.5s

[5/8] ▶  05_Regression_Preparation.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 5.3s

[6/8] ▶  06_Regression_Lineaire.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 9.6s

[7/8] ▶  07_Regression_Comparaison_Modeles.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 15.7s

[8/8] ▶  08_Conclusion.ipynb
------------------------------------------------------------
  ✓ OK  —  durée : 4.3s

  RÉCAPITULATIF